In [0]:
display(dbutils.fs.ls("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/"))

path,name,size,modificationTime
dbfs:/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/,stream_json/,0,1779861521959


In [0]:
display(dbutils.fs.ls("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/"))

path,name,size,modificationTime
dbfs:/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/README.md,README.md,1678,1727470410000
dbfs:/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/customers/,customers/,0,1779861531050
dbfs:/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/,orders/,0,1779861531050
dbfs:/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/status/,status/,0,1779861531050


In [0]:
# created a static order. 

static_orders = spark.read.option(
    "multiLine",
    "true"
).json(
    "/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/"
)

display(static_orders)

customer_id,notifications,order_id,order_timestamp
23094,Y,75123,1640392092
23457,N,75124,1640392500
23564,Y,75125,1640394862
23392,N,75126,1640396067
23101,Y,75127,1640399066
23466,N,75128,1640404853
23834,Y,75129,1640407272
23852,Y,75130,1640419989
23483,Y,75131,1640422131
23821,N,75132,1640423697


In [0]:
static_orders.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- notifications: string (nullable = true)
 |-- order_id: long (nullable = true)
 |-- order_timestamp: long (nullable = true)



In [0]:
spark.sql("""create volume if not exists retail_project.streaming_volume""")

DataFrame[]

In [0]:
spark.sql(""" show volumes in retail_project""").show()

+--------------+----------------+
|      database|     volume_name|
+--------------+----------------+
|retail_project|streaming_volume|
+--------------+----------------+



In [0]:
spark.sql("select current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+



In [0]:
spark.sql(""" create volume if not exists workspace.retail_project.streaming_volume""")

DataFrame[]

In [0]:
spark.sql(""" show volumes in workspace.retail_project""").show()

+--------------+----------------+
|      database|     volume_name|
+--------------+----------------+
|retail_project|streaming_volume|
+--------------+----------------+



In [0]:
streaming_orders = spark.readStream \
    .schema(static_orders.schema) \
    .json(
        "/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/"
    )

In [0]:
display(
    streaming_orders,
    checkpointLocation="/Volumes/workspace/retail_project/streaming_volume/display_orders_checkpoint"
)

Checkpointing to /Volumes/workspace/retail_project/streaming_volume/display_orders_checkpoint


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6461037962190060>, line 1
----> 1 display(
      2     streaming_orders,
      3     checkpointLocation="/Volumes/workspace/retail_project/streaming_volume/display_orders_checkpoint"
      4 )

File /databricks/python_shell/lib/dbruntime/display.py:136, in Display.display(self, input, *args, **kwargs)
    134     pass
    135 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 136     self.display_connect_table(input, **kwargs)
    137 elif isinstance(input, ConnectDataFrame):
    138     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:97, in Display.display_connect_table(self, df, **kwargs)
     92     raise type(
     93         e
     94     )("IPython shell encountered an error or was missing data, please restart the notebook or contact Databricks support"
     95

In [0]:
stream_query = streaming_orders.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .option(
        "checkpointLocation",
        "/Volumes/workspace/retail_project/streaming_volume/streaming_orders_checkpoint"
    ) \
    .table("retail_project.streaming_orders")

In [0]:
display(spark.table("retail_project.streaming_orders"))

In [0]:
spark.sql("""
SELECT COUNT(*) AS total_stream_records
FROM retail_project.streaming_orders
""").show()

In [0]:
spark.table(
    "retail_project.streaming_orders"
).printSchema()